# Chunk-shape sensitivity

Fixed `N = 500_000` point cloud uniformly distributed in `[0, 1000)³`.
Sweep `chunk_shape` ∈ `{50, 100, 200, 400, 800}` (uniform 3-tuples) and
measure write, full read, **bbox read** (1% of volume), disk size, and
chunk count.  `bin_shape = chunk_shape / 4` for every run.

The bbox panel is the headline.  Expect a U-shape: tiny chunks pay
per-file metadata overhead, huge chunks waste I/O on data the bbox
didn't ask for.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.lazy import open_zv

N           = 500_000
CHUNK_SIZES = [50, 100, 200, 400, 800]
DOMAIN      = 1000.0
BBOX_V      = 0.01           # 1% volume read-bbox per run
SEED        = 0

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)
positions = rng.uniform(0, DOMAIN, (N, 3)).astype(np.float32)

metrics = ['write_s', 'read_all_s', 'read_bbox_s']
raw = {m: np.zeros((len(CHUNK_SIZES), N_RUNS)) for m in metrics}
sizes_MB    = np.zeros(len(CHUNK_SIZES))
chunk_count = np.zeros(len(CHUNK_SIZES))

for i, cs in enumerate(CHUNK_SIZES):
    chunk_shape = (float(cs),) * 3
    bin_shape   = (cs / 4.0,) * 3
    edge = DOMAIN * (BBOX_V ** (1 / 3))

    for run in range(N_RUNS):
        store = _new_store(f'cs_{cs}_run{run}')

        t_w, _ = _time(
            write_points, store, positions,
            chunk_shape=chunk_shape, bin_shape=bin_shape,
        )
        t_r, _ = _time(read_points, store)

        lo = rng.uniform(0, DOMAIN - edge, 3).astype(np.float32)
        hi = (lo + edge).astype(np.float32)
        zv = open_zv(store)
        t_b, _ = _time(lambda: zv[0].filter(bbox=(lo, hi)).compute())

        raw['write_s'][i, run]     = t_w
        raw['read_all_s'][i, run]  = t_r
        raw['read_bbox_s'][i, run] = t_b

        if run == 0:
            sizes_MB[i]    = _store_bytes(store) / 1e6
            chunk_count[i] = len(zv[0].chunk_keys)

        shutil.rmtree(Path(store).parent, ignore_errors=True)

rows = []
for i, cs in enumerate(CHUNK_SIZES):
    row = {'chunk_shape': cs, 'size_MB': round(sizes_MB[i], 2),
           'chunk_count': int(chunk_count[i])}
    for m in metrics:
        mean, hw = _mean_ci95(raw[m][i])
        row[f'{m}_mean'] = round(mean, 4)
        row[f'{m}_hw']   = round(hw,   4)
    rows.append(row)
df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

def _line(ax, key, ylabel, title, color):
    mean = df[f'{key}_mean'].values
    hw   = df[f'{key}_hw'].values
    ax.fill_between(df['chunk_shape'], mean - hw, mean + hw,
                    color=color, alpha=0.2)
    ax.loglog(df['chunk_shape'], mean, 'o-', color=color)
    ax.set_xlabel('chunk_shape (per axis)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)

_line(axes[0, 0], 'write_s',     's', 'Write time',       'C0')
_line(axes[0, 1], 'read_all_s',  's', 'Read all',         'C1')
_line(axes[1, 0], 'read_bbox_s', 's', f'Read bbox (V={BBOX_V*100}%)', 'C2')

# Disk size + chunk count on the same axis.
ax = axes[1, 1]
ax.loglog(df['chunk_shape'], df['size_MB'], 'o-', color='C3', label='disk MB')
ax.set_xlabel('chunk_shape (per axis)')
ax.set_ylabel('MB')
ax.grid(True, which='both', alpha=0.3)
ax_r = ax.twinx()
ax_r.loglog(df['chunk_shape'], df['chunk_count'], 's--', color='C4',
            label='chunk count')
ax_r.set_ylabel('chunk count')
ax.set_title('Disk MB and chunk count')
ax.legend(loc='upper left')
ax_r.legend(loc='lower right')

fig.suptitle(
    f'Chunk-shape sensitivity — point cloud (N = {N:,}, '
    f'{N_RUNS} runs, 95% CI)',
)
plt.tight_layout()